## Restore runner (headless)

Previews or runs one lakehouse restore with **no widgets and no typed names**, so it can be started by the Fabric **Run On Demand Notebook Job** API. It does the same thing as `nb_restore_form` + `nb_restore_executor`, on the same engine (`nb_backup_restore_adls_v6`, loaded with `%run`, so it **must be in this same workspace**).

- `mode = "preview"` (default) is **read only**: it returns what is in the way and, if cleanup was asked for and confirmed, what would be deleted.
- `mode = "restore"` with `dry_run = True` (default) runs every check of a real restore and still changes nothing. A real restore needs `mode = "restore"` **and** `dry_run = False`.
- Guards: the backup must be complete (manifest), `clean_target` of `replace` or `all` needs `confirm_target_name` to equal the resolved target lakehouse name **exactly**, `all` cannot be combined with schema/table filters, the container must be readable.
- **Result**: one JSON document via `notebookutils.notebook.exit(...)`, printed as `RESULT_JSON ...`, optionally also written to a control lakehouse. A failed run ends the job as *Failed* unless `fail_on_error` is false.

In [ ]:
# Parameters. This cell is tagged "parameters": the values are defaults, and the Run On Demand job API
# (or a pipeline / notebookutils.notebook.run) overrides them by name. Names are case-sensitive in the code.
# Send IDs, not names. Types in the job request: Text -> str, Boolean -> bool, Integer -> int.
# The defaults are the safe ones: preview only, nothing deleted.
mode = "preview"                  # Text   "preview" (read only) or "restore"
archive_root_uri = ""             # Text   abfss://<container>@<account>.dfs.core.windows.net
backup_folder = ""                # Text   folder inside the container that holds backups/ ("" = the container root)
backup_timestamp = ""             # Text   12 digits YYYYMMDDHHMM of a COMPLETE backup (required)
target_workspace_id = ""          # Text   GUID of the workspace that holds the lakehouse to restore into
target_lakehouse_id = ""          # Text   GUID of the lakehouse to restore into
restore_files = True              # Boolean  also restore everything under Files (form default: on)
only_schemas = ""                 # Text   optional, comma separated; blank = all
only_tables = ""                  # Text   optional, comma separated; blank = all
clean_target = ""                 # Text   "" = stop if anything is in the way | "replace" | "all"
confirm_target_name = ""          # Text   required for "replace"/"all" when mode="restore": the exact target lakehouse name
dry_run = True                    # Boolean  with mode="restore": True = run every check and show the plan, change nothing
request_key = ""                  # Text   caller's id for tracing; blank = a manual-xxxx id is made
result_workspace_id = ""          # Text   optional: workspace of a control lakehouse to receive the result JSON file
result_lakehouse_id = ""          # Text   optional: control lakehouse (Files/rayfin_runs/...); must not be the target
fail_on_error = True              # Boolean  True: a failed run ends the job as Failed (JSON in the error text); False: always exit with JSON

### Load the engine (same workspace)

In [ ]:
%run nb_backup_restore_adls_v6

### Helpers

In [ ]:
# ---- shared runner helpers (identical in nb_backup_run and nb_restore_run; engine is not touched) ----
import contextlib
import json
import logging
import re
import sys
import uuid
from datetime import datetime, timezone

RUNNER = "nb_restore_run"
SCHEMA_VERSION = 1
LIST_CAP = 200                       # long lists in the exit JSON are cut here (the manifest in the archive has all)
_STARTED = datetime.now(timezone.utc)
_GUID = re.compile(r"[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}")


class RunnerError(Exception):
    """A problem the caller can act on. `code` is stable and machine-readable."""
    def __init__(self, code, message):
        super().__init__(message)
        self.code = code
        self.message = message


def _guid(value, label):
    text = str(value).strip().lower()
    if not _GUID.fullmatch(text):
        raise RunnerError("invalid_parameter", f"{label} must be a GUID, got {str(value)!r}")
    return text


def _bool(value, label):
    """Accepts real booleans; also the strings 'true'/'false' in case a caller sent Text instead of Boolean."""
    if isinstance(value, bool):
        return value
    if isinstance(value, str) and value.strip().lower() in ("true", "false"):
        return value.strip().lower() == "true"
    raise RunnerError("invalid_parameter", f"{label} must be true or false, got {value!r}")


def _request_key(value):
    text = str(value).strip()
    if not text:
        text = "manual-" + uuid.uuid4().hex[:8]
    if not re.fullmatch(r"[A-Za-z0-9._:-]{1,128}", text):
        raise RunnerError("invalid_parameter", "request_key may only contain letters, digits, . _ : - (max 128)")
    return text


def _archive_uri(archive_root_uri, backup_folder):
    """Same address rule as the forms (abfss form only) plus the optional single folder."""
    uri = str(archive_root_uri).strip().rstrip("/")
    if not re.fullmatch(r"abfss://[^@/\s]+@[^./\s]+\.dfs\.core\.windows\.net(/\S*)?", uri):
        raise RunnerError("invalid_parameter",
                          "archive_root_uri must look like abfss://<container>@<account>.dfs.core.windows.net")
    folder = str(backup_folder).strip().strip("/")
    if folder:
        for part in folder.split("/"):
            if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9._-]*", part):
                raise RunnerError("invalid_parameter",
                                  "backup_folder may only contain letters, digits, dot, dash and underscore")
        uri = f"{uri}/{folder}"
    if any(part in (".", "..") for part in uri.split("//", 1)[1].split("/")):
        raise RunnerError("invalid_parameter", "archive_root_uri must not contain . or .. segments")
    return uri


def _name_set(text, label):
    """'dbo, sales' -> {'dbo', 'sales'}. Blank -> None, which means everything."""
    names = {part.strip() for part in str(text).split(",") if part.strip()}
    return names or None


def _container_reachable(archive):
    """Blocking version of the form's check: the container itself must be listable."""
    root = re.match(r"abfss://[^/]+", archive).group(0)
    try:
        fs.ls(root)
    except Exception as exc:
        raise RunnerError("container_unreachable",
                          f"Cannot read the container {root}. Check the address and the runner's access. "
                          + " ".join(str(exc).split())[:200])


def _lakehouse(workspace_id, lakehouse_id, label):
    """Engine lookup (Fabric REST, runner identity). Proves the IDs exist and are readable."""
    try:
        return lakehouse_info(workspace_id, lakehouse_id)
    except Exception as exc:
        raise RunnerError("lakehouse_not_found",
                          f"{label}: workspace {workspace_id} / lakehouse {lakehouse_id} was not found, "
                          f"or the runner has no access to it. {' '.join(str(exc).split())[:200]}")


def _cap(items):
    items = list(items)
    return items[:LIST_CAP], len(items)


class _Tee:
    """Copies what the engine prints into a list while still passing it through."""
    def __init__(self, stream):
        self._stream, self.lines, self._buf = stream, [], ""

    def write(self, text):
        self._buf += text
        while "\n" in self._buf:
            line, self._buf = self._buf.split("\n", 1)
            self.lines.append(line)
        return self._stream.write(text)

    def flush(self):
        return self._stream.flush()

    def __getattr__(self, name):
        return getattr(self._stream, name)


class _LogCollector(logging.Handler):
    """Collects the engine's log messages (its logger is `logger`) so a failure can report where it stopped."""
    def __init__(self):
        super().__init__(logging.INFO)
        self.messages = []

    def emit(self, record):
        self.messages.append(f"{record.levelname} {record.getMessage()}")


@contextlib.contextmanager
def _capture():
    """Capture engine prints and log lines for the duration of one engine call."""
    tee, collector = _Tee(sys.stdout), _LogCollector()
    logger.addHandler(collector)
    try:
        with contextlib.redirect_stdout(tee):
            yield tee, collector
    finally:
        logger.removeHandler(collector)


def _base_result():
    return {"schema_version": SCHEMA_VERSION, "runner": RUNNER, "request_key": None,
            "status": "failed", "error": None,
            "started_utc": _STARTED.isoformat(timespec="seconds")}


def _failure(result, code, message, log=None, report=None):
    result["status"] = "failed"
    result["error"] = {"code": code, "message": str(message)[:1500]}
    if log:
        result["log_tail"] = list(log)[-25:]
    if report:
        result["report"] = list(report)[-25:]
    return result


def _finish(result):
    finished = datetime.now(timezone.utc)
    result["finished_utc"] = finished.isoformat(timespec="seconds")
    result["duration_seconds"] = round((finished - _STARTED).total_seconds(), 1)
    return result


def _write_result_file(result, result_workspace_id, result_lakehouse_id):
    """Optional: the job API does not return the exit value, so the runner can also drop the JSON in a
    control lakehouse the app can read: Files/rayfin_runs/<runner>/<request_key>.json. Best effort."""
    if not (str(result_workspace_id).strip() and str(result_lakehouse_id).strip()):
        return None
    try:
        path = (f"{lakehouse_root(_guid(result_workspace_id, 'result_workspace_id'), _guid(result_lakehouse_id, 'result_lakehouse_id'))}"
                f"/Files/rayfin_runs/{RUNNER}/{result['request_key']}.json")
        fs.put(path, json.dumps(result, default=str, indent=2), True)
        return path
    except Exception as exc:
        logger.warning("Could not write the result file: %s", reason(exc))
        return None

### Validation (the form's guards, server side)

In [ ]:
def validate_restore_params():
    """Server-side version of the Restore form's guards. Reads only. Returns the clean values to run with."""
    v = {}
    v["mode"] = str(mode).strip().lower()
    if v["mode"] not in ("preview", "restore"):
        raise RunnerError("invalid_parameter", f'mode must be "preview" or "restore", got {mode!r}')
    v["dry_run"] = _bool(dry_run, "dry_run")
    v["fail_on_error"] = _bool(fail_on_error, "fail_on_error")
    v["restore_files"] = _bool(restore_files, "restore_files")
    v["tgt_ws"] = _guid(target_workspace_id, "target_workspace_id")
    v["tgt_lh"] = _guid(target_lakehouse_id, "target_lakehouse_id")
    v["archive"] = _archive_uri(archive_root_uri, backup_folder)
    ts = str(backup_timestamp).strip()
    if not re.fullmatch(r"\d{12}", ts):
        raise RunnerError("invalid_parameter", f"backup_timestamp must be 12 digits (YYYYMMDDHHMM), got {ts!r}")
    v["ts"] = ts
    v["schemas"] = _name_set(only_schemas, "only_schemas")
    v["tables"] = _name_set(only_tables, "only_tables")
    clean = str(clean_target).strip().lower()
    if clean not in ("", "replace", "all"):
        raise RunnerError("invalid_parameter", f'clean_target must be "", "replace" or "all", got {clean_target!r}')
    v["clean"] = clean or False                               # the engine's values: False | "replace" | "all"
    if v["clean"] == "all" and (v["schemas"] is not None or v["tables"] is not None):
        raise RunnerError("invalid_parameter",
                          'clean_target "all" cannot be combined with only_schemas / only_tables; use "replace"')
    if str(result_workspace_id).strip() or str(result_lakehouse_id).strip():
        rw, rl = _guid(result_workspace_id, "result_workspace_id"), _guid(result_lakehouse_id, "result_lakehouse_id")
        if (rw, rl) == (v["tgt_ws"], v["tgt_lh"]):
            raise RunnerError("invalid_parameter", "The result lakehouse must not be the restore target.")
    _container_reachable(v["archive"])
    # the backup must be complete: it has a manifest that says so
    if not fs.exists(manifest_path_of(v["archive"], ts)):
        raise RunnerError("backup_incomplete",
                          f"Backup {ts} has no manifest in {v['archive']}/backups: it is incomplete or does not exist.")
    v["manifest"] = read_json(manifest_path_of(v["archive"], ts))
    if v["manifest"].get("status") != "complete":
        raise RunnerError("backup_incomplete", f"The manifest of backup {ts} does not say complete.")
    v["info"] = _lakehouse(v["tgt_ws"], v["tgt_lh"], "Target lakehouse")
    confirm = str(confirm_target_name).strip()
    v["confirm"] = confirm if v["clean"] else None
    # exact, case-sensitive match against the RESOLVED name (same rule as the form and the engine)
    v["confirm_matches"] = (confirm == v["info"]["lakehouse_name"]) if v["clean"] else None
    if v["mode"] == "restore" and v["clean"] and not v["confirm_matches"]:
        raise RunnerError("confirmation_mismatch",
                          "Deleting from the target was chosen. confirm_target_name must equal the target lakehouse "
                          f"name exactly: {v['info']['lakehouse_name']}")
    return v

### Run

In [ ]:
RESULT = _base_result()
_report, _log = [], []
_restore_started = False
try:
    RESULT["request_key"] = _request_key(request_key)
    V = validate_restore_params()
    _m, _info = V["manifest"], V["info"]
    _preview_only = V["mode"] == "preview" or V["dry_run"]
    RESULT.update({
        "mode": V["mode"], "dry_run": V["dry_run"] if V["mode"] == "restore" else None,
        "modified_target": False,
        "backup_timestamp": V["ts"], "archive": V["archive"],
        "target": {"workspace_id": V["tgt_ws"], "lakehouse_id": V["tgt_lh"],
                   "workspace_name": _info["workspace_name"], "lakehouse_name": _info["lakehouse_name"],
                   "has_schemas": _info["has_schemas"]},
        "backup": {"source_workspace_name": _m["source"]["workspace_name"],
                   "source_lakehouse_name": _m["source"]["lakehouse_name"],
                   "completed_utc": _m.get("completed_utc"), "tables_in_backup": len(_m.get("tables", [])),
                   "files_in_backup": bool(_m.get("files_copied"))},
        "restore_files": V["restore_files"], "clean_target": V["clean"] or "",
        "only_schemas": sorted(V["schemas"]) if V["schemas"] else [],
        "only_tables": sorted(V["tables"]) if V["tables"] else [],
        "confirm_matches": V["confirm_matches"],
    })
    _chosen = [t for t in _m.get("tables", [])
               if (V["schemas"] is None or t["schema"] in V["schemas"])
               and (V["tables"] is None or t["table"] in V["tables"])]
    RESULT["tables_to_restore"] = len(_chosen)
    RESULT["unmatched_schemas"] = sorted((V["schemas"] or set()) - {t["schema"] for t in _m.get("tables", [])})
    RESULT["unmatched_tables"] = sorted((V["tables"] or set()) - {t["table"] for t in _m.get("tables", [])})

    # ---- read-only look at the target (also the first half of a real restore) ---------------------------
    with _capture() as (_tee, _logs):
        try:
            _clashes = preview_restore(
                V["archive"], V["ts"], V["tgt_ws"], V["tgt_lh"],
                schemas=V["schemas"], tables=V["tables"],
                restore_files=V["restore_files"], clean_target=V["clean"],
            )
        finally:
            _report, _log = _tee.lines, _logs.messages
    RESULT["clashes"], RESULT["clashes_count"] = _cap(_clashes)
    RESULT["outcome"] = ("nothing_in_the_way" if not _clashes else
                         "will_stop_target_not_empty" if not V["clean"] else
                         "will_delete_then_restore" if V["clean"] == "replace" else
                         "will_delete_everything_then_restore")
    RESULT["cleanup_plan"], RESULT["cleanup_plan_error"] = None, None
    if V["clean"] and V["confirm_matches"]:
        try:                                                   # dry_run=True: lists, deletes nothing
            _plan = clean_target_for_restore(
                V["archive"], V["ts"], V["tgt_ws"], V["tgt_lh"],
                mode=V["clean"], confirm_target_name=V["confirm"],
                include_files=V["restore_files"], schemas=V["schemas"], tables=V["tables"], dry_run=True,
            )
            RESULT["cleanup_plan"], RESULT["cleanup_plan_count"] = _cap(_plan)
        except Exception as exc:
            RESULT["cleanup_plan_error"] = reason(exc)

    if _preview_only:
        RESULT["status"] = "succeeded"
        RESULT["report"] = _report[-20:]
    else:
        if _clashes and not V["clean"]:
            raise RunnerError("target_not_empty",
                              f"{len(_clashes)} item(s) already exist in the target and clean_target is not set; "
                              "nothing was restored. First clashes: " + ", ".join(c["path"] for c in _clashes[:5]))
        if RESULT["cleanup_plan_error"]:
            raise RunnerError("cleanup_blocked", RESULT["cleanup_plan_error"])
        _restore_started = True
        with _capture() as (_tee, _logs):
            try:
                _results = restore_from_archive(
                    V["archive"], V["ts"],
                    tgt_ws_id=V["tgt_ws"], tgt_lh_id=V["tgt_lh"],
                    schemas=V["schemas"], tables=V["tables"],
                    restore_files=V["restore_files"],
                    clean_target=V["clean"],
                    confirm_target_name=V["confirm"],
                )
            finally:
                _report, _log = _tee.lines, _logs.messages
        RESULT["modified_target"] = True
        _rows = [{"schema": r["schema"], "table": r["table"], "version": r["version"],
                  "outcome": "restored" if r["status"] == "restored" else "copied_registration_failed",
                  "status": r["status"], "location": r["location"]} for r in _results]
        _bad = [r for r in _rows if r["outcome"] != "restored"]
        RESULT["tables"], RESULT["tables_count"] = _cap(_rows)
        RESULT["tables_restored"] = len(_rows) - len(_bad)
        RESULT["tables_registration_failed"] = len(_bad)
        RESULT["files_restored"] = bool(V["restore_files"] and _m.get("files_copied"))
        RESULT["status"] = "completed_with_problems" if _bad else "succeeded"
        RESULT["report"] = _report[-10:]
except RunnerError as exc:
    _failure(RESULT, exc.code, exc.message, _log, _report)
except Exception as exc:
    if _restore_started:
        RESULT["modified_target"] = "unknown"                  # the restore may have stopped part way
    _failure(RESULT, "restore_failed", reason(exc), _log, _report)
_finish(RESULT)

### Return the result

In [ ]:
# Result delivery. Do not wrap notebookutils.notebook.exit in try/except: the exit would not take effect.
_result_file = (_write_result_file(RESULT, result_workspace_id, result_lakehouse_id)
                if RESULT.get("request_key") else None)
if _result_file:
    RESULT["result_file"] = _result_file
_payload = json.dumps(RESULT, default=str, separators=(",", ":"))
print("RESULT_JSON " + _payload)
if RESULT["status"] == "failed" and str(fail_on_error).strip().lower() != "false":
    raise RuntimeError("RAYFIN_RESULT " + _payload)          # job ends as Failed; failureReason carries the JSON
notebookutils.notebook.exit(_payload)